# Участки СОЗ‑пестицидов: скрининг опасности и индекс риска для детей

Демонстрационный блокнот пилотного проекта. Здесь на реальных данных воспроизводятся два метода стенда:

- **модель скрининга** — оценивает, вероятно ли сильное загрязнение участка, ещё до лабораторного анализа;
- **индекс риска для детей** — по логике детского индекса климатических и экологических рисков UNICEF (CCRI‑DRM):
  риск = опасность × дети рядом × пути воздействия.

**Что внутри**

1. Откуда данные и что в них есть.
2. Насколько загрязнены участки.
3. Модель скрининга: признаки, пространственная проверка, кривая отбора, проверка на случайность.
4. Что влияет на оценку модели.
5. Оценка участка без анализов и вопрос «что если».
6. Индекс риска для детей и его устойчивость к выбору весов.
7. План следующей кампании отбора проб.

**Как читать.** Код скрыт; кнопка «Показать код» вверху страницы раскрывает его. Все расчёты выполняются локально из
файлов слоя `source_data/chemicals_tj/pop_layer/`, без базы данных и без интернета.

In [ ]:
import sys, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import demo_utils as U
from demo_utils import f, pct
U.setup()
sys.path.insert(0, str(U.ROOT / 'db'))
import pop_screening_model as M
LAYER = U.DATA / 'source_data/chemicals_tj/pop_layer'
sites = pd.read_csv(LAYER / 'pop_sites.csv', encoding='utf-8-sig', low_memory=False)
samples = pd.read_csv(LAYER / 'pop_samples.csv', encoding='utf-8-sig', low_memory=False)
risk = pd.read_csv(LAYER / 'child_risk/child_risk_index.csv', encoding='utf-8-sig')
T, L, targets = M.get_data()
y = targets[M.FINAL_TARGET]
print(f"Участков на карте: {len(T)}, с анализами ДДТ/ГХЦГ: {len(L)}, из них сильно загрязнены (≥100 мг/кг): {y.sum()}")

## 1. Откуда данные

Структурированных таблиц у заказчика нет. Слой собран из портала chemicals.tj: карта участков, анкеты обследования,
текст карточек и прикреплённые документы — лабораторные протоколы, журналы отбора проб, отсканированные отчёты.
Значения извлечены из PDF и сверены со сканами (совпало 229 из 233 проверенных наборов).

In [ ]:
src = pd.read_csv(LAYER / 'pop_sources.csv', encoding='utf-8-sig')
names = {'AFI126_2019': 'Агрофизический НИИ, протокол №126 (2019, скан)', 'AFI8_2021': 'Агрофизический НИИ, протокол №8 (2020)',
         'ILIM42_2017': 'НПО «ИЛИМ», отчёт №42 (2016)', 'JOURNAL': 'журналы и реестры отбора проб',
         'KUKANIEV_2012': 'Институт химии, таблица 2012 г.', 'NIKITIN_2017': 'Институт химии, письмо 2017 г. (скан)'}
t = pd.DataFrame({'Источник': src.source_id.map(names), 'Значений в сводной выборке': src.rank_1_2,
                  'Участков': src.n_sites}).sort_values('Значений в сводной выборке', ascending=False)
U.table(t)
U.note(f"Итого: {len(sites)} участков (192 на карте и {int((sites.on_map != True).sum())} только в протоколах), "
       f"{len(samples)} проб, {f(src.rank_1_2.sum(), 0)} значения концентраций без дублей.")

In [ ]:
R = risk.copy()
bins = [-1, .1, 10, 100, 1e9]; labs = ['≤ 0,1 (норматив)', '0,1–10', '10–100', '≥ 100 мг/кг']
R['cls'] = np.where(R.burden.isna(), 'нет анализов', pd.cut(R.burden, bins, labels=labs).astype(str))
colors = {'нет анализов': '#ffffff', '≤ 0,1 (норматив)': U.GREEN, '0,1–10': '#e3b23c', '10–100': U.ORANGE, '≥ 100 мг/кг': U.RED}
fig, axs = plt.subplots(1, 2, figsize=(11, 5.2), gridspec_kw={'width_ratios': [1, 1.25]})
for ax, (b, title) in zip(axs, [((67.3, 36.6, 71.2, 41.1), 'Все участки карты'),
                               ((67.8, 37.1, 69.9, 38.4), 'Юг Хатлонской области (крупнее)')]):
    U.hillshade(ax, b)
    for k, c in colors.items():
        d = R[R.cls == k]
        ax.scatter(d.longitude, d.latitude, s=28 if k != '≥ 100 мг/кг' else 40, c=c, edgecolor='#333', lw=.5,
                   label=f'{k} ({len(d)})', zorder=3)
    U.map_axes(ax); ax.set_title(title)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, title='ДДТ‑группа или ГХЦГ, максимум по пробам, мг/кг', fontsize=9, title_fontsize=9,
           loc='lower center', ncol=5, bbox_to_anchor=(.5, -.06))
plt.show()

## 2. Насколько загрязнены участки

Показатель загрязнения участка — максимум по пробам из двух сумм: ДДТ+ДДД+ДДЕ и изомеров ГХЦГ, мг/кг. Норматив ДДТ
в почве — 0,1 мг/кг. Шкала логарифмическая: каждое деление — в 10 раз больше.

In [ ]:
b = L.burden.clip(lower=.001)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(np.log10(b), bins=np.arange(-3, 4.01, .25), color=U.GREY, edgecolor='white')
ax.hist(np.log10(b[b >= 100]), bins=np.arange(-3, 4.01, .25), color=U.RED, edgecolor='white', label='сильно загрязнены (≥100 мг/кг)')
for v, lab in [(.1, 'норматив 0,1'), (10, '×100'), (100, '×1000')]:
    ax.axvline(np.log10(v), color='#333', ls='--', lw=1); ax.text(np.log10(v) + .05, ax.get_ylim()[1]*.92, lab, fontsize=9)
ax.set_xticks(range(-3, 5), ['0,001', '0,01', '0,1', '1', '10', '100', '1 000', '10 000'])
ax.annotate('не обнаружено', (-2.87, 4.3), ha='center', fontsize=8, color='#555')
ax.set_xlabel('максимум по пробам участка, мг/кг'); ax.set_ylabel('участков'); ax.legend(loc='upper right')
ax.set_title(f'Загрязнение {len(L)} обследованных участков'); plt.show()
U.note(f"Норматив превышен на {int((L.burden > .1).sum())} участках из {len(L)}, в 100 раз — на {int((L.burden >= 10).sum())}, "
       f"в 1000 раз — на {int((L.burden >= 100).sum())}. Максимум — {f(L.burden.max(), 0)} мг/кг.")

## 3. Модель скрининга

**Вопрос:** можно ли по сведениям, которые есть до лабораторного анализа, отличить сильно загрязнённые участки
(≥100 мг/кг, в 1000 раз выше норматива) от остальных?

**Признаки** — только то, что известно до анализа:

- анкета обследования chemicals.tj (площадь, запах, яма, доступ, покрытие почвы, посевы, животные, вода, жители);
- место: область и координаты;
- загрязнение пяти ближайших обследованных участков;
- открытые данные из локальных растров: pH почвы, высота и уклон, население, накопление стока, температура и осадки.

**Алгоритм:** логистическая регрессия с регуляризацией — простая и объяснимая модель.

**Пространственная проверка.** Соседние участки похожи друг на друга, поэтому обычная проверка завышает качество. Здесь
участки делятся на 10 географических кластеров, и каждый кластер по очереди целиком исключается из обучения.
Ниже — одна из десяти таких разбивок.

In [ ]:
from sklearn.cluster import KMeans
gr = KMeans(M.N_CLUSTERS, n_init=5, random_state=0).fit_predict(L[['latitude', 'longitude']].values)
fig, ax = plt.subplots(figsize=(7.5, 5))
U.hillshade(ax, (67.3, 36.6, 71.2, 41.1))
ax.scatter(L.longitude, L.latitude, c=gr, cmap='tab10', s=30, edgecolor='#333', lw=.4, zorder=3)
U.map_axes(ax); ax.set_title('10 пространственных кластеров: модель проверяется на кластере, которого не видела')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score
FS = "G. Анкета + регион/координаты + соседи + ковариаты"
cats, nums = M.FEATURE_SETS[FS]
oof = M.spatial_oof(L, y, cats, nums, "Логистическая регрессия")     # 10 разбивок × 10 кластеров
aucs = [roc_auc_score(y, o) for o in oof]
score = oof.mean(0); auc = roc_auc_score(y, score)
U.note(f"AUC вне выборки: {f(auc, 2)} (по десяти разбивкам — от {f(min(aucs), 2)} до {f(max(aucs), 2)}). "
       f"AUC = 0,5 — угадывание, 1 — безошибочное разделение.")

In [ ]:
order = np.argsort(-score, kind='stable'); found = np.cumsum(y[order]); k = np.arange(1, len(y) + 1)
rnd = k * y.mean(); ideal = np.minimum(k, y.sum())
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(k, ideal, color=U.GREY, ls=':', lw=1.2, label='идеальный отбор')
ax.plot(k, rnd, color=U.GREY, ls='--', lw=1.6, label='случайный выбор')
ax.plot(k, found, color=U.BLUE, lw=2.5, label='отбор по модели (проверка вне выборки)')
for kk in (20, 30):
    ax.plot(kk, found[kk-1], 'o', color=U.BLUE)
    ax.annotate(f'{kk} участков: {found[kk-1]} против {f(rnd[kk-1], 1)} при случайном выборе', (kk, found[kk-1]),
                xytext=(60, 9 if kk == 30 else 3), textcoords='data', fontsize=9,
                arrowprops=dict(arrowstyle='-', color='#777'))
ax.set_xlabel('сколько участков обследовано (по убыванию оценки модели)'); ax.set_ylabel('найдено сильно загрязнённых')
ax.set_title('Кривая отбора: модель находит сильно загрязнённые участки вдвое быстрее')
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.16), ncol=3)
plt.show()

### Проверка на случайность

Может ли такой результат получиться случайно? Метки «загрязнён / не загрязнён» перемешиваются, и вся процедура
проверки повторяется. Если модель ничего не нашла, её AUC будет типичным для перемешанных данных. Для демонстрации
здесь 40 перестановок (по 3 разбивки); полный расчёт пилота — 304 перестановки, p = 0,003.

In [ ]:
rng = np.random.default_rng(1); M.N_REPEATS = 3
obs = roc_auc_score(y, M.spatial_oof(L, y, cats, nums, "Логистическая регрессия").mean(0))
null = [roc_auc_score(yp, M.spatial_oof(L, yp, cats, nums, "Логистическая регрессия").mean(0))
        for yp in (rng.permutation(y) for _ in range(40))]
M.N_REPEATS = 10
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(null, bins=12, color=U.GREY, edgecolor='white', label='перемешанные метки')
ax.axvline(obs, color=U.BLUE, lw=2.5, label=f'настоящие данные: {f(obs, 2)}')
ax.set_xlabel('AUC'); ax.set_ylabel('перестановок'); ax.legend(); U.comma_axis(ax, 'x', 2)
ax.set_title('Результат модели против случайности'); plt.show()
U.note(f"Ни одна из 40 перестановок не достигла AUC настоящих данных (максимум среди перестановок — {f(max(null), 2)}). "
       if max(null) < obs else f"{sum(n >= obs for n in null)} из 40 перестановок достигли AUC настоящих данных.")

### Какие сведения полезны

Модель проверялась на нескольких наборах признаков (результаты полного прогона пилота, та же проверка).

In [ ]:
m = pd.read_csv(LAYER / 'model/metrics.csv', encoding='utf-8-sig')
m = m[m.target.str.contains('основная') & (m.model == 'Логистическая регрессия')].copy()
lab = {'A. Анкета': 'анкета', 'B. Анкета + текст карточки': 'анкета + текст карточки',
       'E. Только регион/координаты + соседи': 'место + соседние участки',
       'F. Открытые ковариаты (pH, рельеф, население, сток, климат)': 'открытые данные (pH, рельеф, население, сток, климат)',
       'G. Анкета + регион/координаты + соседи + ковариаты': 'всё вместе (итоговая модель)'}
m = m[m.features.isin(lab)].assign(name=lambda d: d.features.map(lab)).sort_values('auc_mean')
fig, ax = plt.subplots(figsize=(8, 3.4))
cols = [U.BLUE if n.startswith('всё') else U.GREY for n in m.name]
ax.barh(m.name, m.auc_mean, color=cols, xerr=[m.auc_mean - m.auc_p05, m.auc_p95 - m.auc_mean], ecolor='#555', capsize=3)
for i, v in enumerate(m.auc_mean): ax.text(.505, i, f(v, 2), va='center', color='white', fontsize=9, fontweight='bold')
ax.axvline(.5, color='#333', lw=.8); ax.set_xlim(.5, .8); ax.set_xlabel('AUC вне выборки'); U.comma_axis(ax, 'x', 2)
ax.set_title('Вклад групп сведений'); plt.show()
U.note("Открытые данные о местности сами по себе работают не хуже анкеты. Текст карточек качество снижает "
       "(слишком много признаков для 169 участков), поэтому в модель не входит. Итоговый набор выбран заранее и "
       "не перевыбирался после пересчёта, чтобы не подгонять выбор под результат.")

## 4. Что влияет на оценку модели

Коэффициенты логистической регрессии: положительный — признак повышает оценку, отрицательный — понижает.
Это связь в данных, а не причина: например, «лёгкий доступ» сопутствует старым заброшенным складам.

In [ ]:
NUM = {'ph': 'pH почвы', 'elev_m': 'высота над уровнем моря', 'slope_deg': 'уклон', 'pop_500m': 'население в 500 м',
       'flowacc_max': 'накопление стока', 'bio1_temp_c': 'среднегодовая температура', 'bio12_precip_mm': 'годовые осадки',
       'nb_mean_log_burden': 'загрязнение соседних участков', 'nb_max_log_burden': 'максимум у соседних участков',
       'nb_dist_km': 'расстояние до соседнего обследованного', 'latitude': 'широта', 'longitude': 'долгота',
       'square_ord': 'площадь участка', 'depth': 'глубина ямы', 'lived_on_site': 'живут на участке', 'lived_50m': 'жителей в 50 м',
       'lived_100m': 'жителей в 100 м', 'lived_500m': 'жителей в 500 м', 'river_distance_ord': 'расстояние до реки',
       'groundwater_depth_ord': 'глубина грунтовых вод', 'well_distance_ord': 'расстояние до колодца'}
CAT = {'access': 'доступ', 'status': 'статус', 'agriculture_near': 'посевы', 'land': 'землепользование',
       'animals_food': 'животные и пища', 'surface_water': 'поверхностная вода', 'soil_cover': 'покрытие',
       'water': 'водный объект', 'smell': 'запах', 'pit_dug': 'яма', 'region': 'область', 'flood_zone': 'зона затопления'}
def label(n):
    k, v = n.split('__', 1)
    v = v.replace(', пригодные в пищу', '').replace('Продукты питания и продукты', 'продукты питания')
    if k == 'num':
        if v.startswith('missingindicator_'): return 'нет данных: ' + NUM.get(v[17:], v[17:])
        return NUM.get(v, v)
    for c in sorted(CAT, key=len, reverse=True):
        if v.startswith(c + '_'): return U.short(f"{CAT[c]}: {v[len(c)+1:].lower()}", 52)
    return v
fi = pd.read_csv(LAYER / 'model/feature_importance.csv', encoding='utf-8-sig')
fi = fi[~fi.feature.str.contains('missingindicator|нет данных')]
fi = fi.reindex(fi.weight.abs().sort_values(ascending=False).index).head(10)[::-1]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.barh([label(n) for n in fi.feature], fi.weight, color=[U.BLUE if w > 0 else U.GREY for w in fi.weight])
ax.axvline(0, color='#333', lw=.8); ax.set_xlabel('коэффициент (признаки стандартизованы)'); U.comma_axis(ax, 'x', 1)
ax.set_title('10 самых сильных признаков: синие повышают оценку, серые понижают'); plt.show()

## 5. Оценка участка без анализов и вопрос «что если»

Итоговая модель обучается на всех 169 обследованных участках и оценивает остальные. Оценка переводится в шкалу 0–1
(доля обследованных участков с оценкой ниже) и в класс: нижняя, средняя или верхняя треть. Так работает форма
«Оценить новый участок» на стенде.

In [ ]:
Ltr = M.add_lag(L, L); Tall = M.add_lag(L, T)
model = M.make_model("Логистическая регрессия", cats, nums).fit(Ltr[cats + nums], y)
ref = np.sort(model.predict_proba(Ltr[cats + nums])[:, 1])
r01 = lambda p: np.searchsorted(ref, p) / len(ref)
q1, q2 = np.quantile(M.rank01(score), [1/3, 2/3])
cls = lambda s: 'низкий' if s <= q1 else ('средний' if s <= q2 else 'высокий')
oof_cls = pd.cut(np.round(M.rank01(score), 3), [-1, q1, q2, 2], labels=['низкий', 'средний', 'высокий'])
share = (pd.Series(L.burden.values >= 100).groupby(oof_cls).mean()).to_dict()

un = Tall[T.burden.isna().values].copy()
un['score'] = r01(model.predict_proba(un[cats + nums])[:, 1]); un['class'] = un.score.map(cls)
un = un.merge(risk[['map_id', 'children_1000m']], on='map_id', how='left').sort_values('score', ascending=False)
t = pd.DataFrame({'Код': un.tj_code, 'Участок': un.site_name.map(lambda n: U.short(n, 58)), 'Область': un.region, 'Оценка 0–1': un.score,
                  'Класс': un['class'], 'Детей в 1 км': un.children_1000m})
U.table(t.head(10), nd={'Оценка 0–1': 2, 'Детей в 1 км': 0})
U.note("Доля сильно загрязнённых среди обследованных участков по классам: " +
       ", ".join(f"{c} — {pct(v)}" for c, v in share.items()) + ". Это и есть смысл класса для нового участка.")

In [ ]:
site = un[un.tj_code == 'TJ-7599'].iloc[0] if (un.tj_code == 'TJ-7599').any() else un.iloc[0]
base = Tall[Tall.map_id == site.map_id].copy()
print(f"Пример: {site.tj_code}, {site.site_name}; доступ по анкете: {base.access.iloc[0]}")
rows = [('как в анкете', base.access.iloc[0], r01(model.predict_proba(base[cats + nums])[:, 1])[0])]
for a in [v for v in ['Легкий доступ; малое расстояние для входа', 'Контролируемый доступ; вход затруднен',
                      'Удаленные места; менее доступен'] if v != base.access.iloc[0]]:
    x = base.copy(); x['access'] = a
    rows.append(('если доступ: ' + a.split(';')[0].lower(), a, r01(model.predict_proba(x[cats + nums])[:, 1])[0]))
x = base.copy(); x['nb_mean_log_burden'] = np.log10(0.5)
rows.append(('если соседние участки почти чистые (0,5 мг/кг)', '', r01(model.predict_proba(x[cats + nums])[:, 1])[0]))
w = pd.DataFrame(rows, columns=['Вариант', 'Доступ', 'Оценка 0–1']); w['Класс'] = w['Оценка 0–1'].map(cls)
U.table(w[['Вариант', 'Оценка 0–1', 'Класс']], nd=2)
U.note("Инспектор видит, какие сведения меняют оценку. Так модель подсказывает, что уточнить при выезде.")

## 6. Индекс риска для детей

Индекс отвечает на другой вопрос, чем модель скрининга: не «где грязнее», а **где вмешательство защитит больше детей**.

| Компонента | Как считается |
|---|---|
| Опасность H | log10(загрязнение / 0,1) / 4: норматив → 0, 1000 мг/кг → 1; без анализов — средняя H класса скрининга |
| Дети рядом E | 0,6 × ранг числа детей в 1 км (возрастной растр WorldPop R2025A «до 18 лет», 2025 год) + 0,25 × ранг числа людей на участке + 0,15 × упоминание школы или детей |
| Уязвимость V | 0,7 × доля путей воздействия из семи (питьевая вода, орошение, скот, посевы, свободный доступ, река или паводок ближе 300 м, грунтовые воды ближе 10 м) + 0,3 × бедность по индексу благосостояния Meta (RWI) в радиусе 2 км |
| Риск | (H × E × V)^(1/3), каждая компонента не ниже 0,05 |

In [ ]:
chk = np.cbrt(risk.H.clip(lower=.05) * risk.E.clip(lower=.05) * risk.V.clip(lower=.05))
print('Пересчёт по формуле совпадает с индексом стенда:', bool(np.allclose(chk.round(3), risk.child_risk, atol=.0015)))
top = risk.sort_values('child_risk', ascending=False).head(10)
U.table(pd.DataFrame({'Место': top.child_risk_rank, 'Код': top.tj_code, 'Участок': top.site_name.map(lambda n: U.short(n, 55)),
                      'H': top.H, 'E': top.E, 'V': top.V, 'Риск': top.child_risk,
                      'Детей в 1 км': top.children_1000m}), nd={'H': 2, 'E': 2, 'V': 2, 'Риск': 2, 'Детей в 1 км': 0})
fig, ax = plt.subplots(figsize=(7.5, 4.6))
cc = risk.child_risk_class.map(U.CLASS_COLORS)
ax.scatter(risk.H, risk.E, s=20 + 160*risk.V, c=cc, edgecolor='#333', lw=.4, alpha=.9)
for c, col in U.CLASS_COLORS.items(): ax.scatter([], [], c=col, s=50, label=c, edgecolor='#333', lw=.4)
ax.set_xlabel('опасность H'); ax.set_ylabel('дети рядом E'); U.comma_axis(ax, 'x', 1); U.comma_axis(ax, 'y', 1)
ax.legend(title='класс риска (размер — пути V)', fontsize=8, title_fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
ax.set_title('Высокий риск — там, где сходятся загрязнение, дети и пути воздействия'); plt.show()

### Устойчивость к выбору весов

Веса компонент — экспертное решение. Чтобы показать, насколько от него зависит результат, индекс пересчитывается
1000 раз со случайными весами H, E и V (в среднем по трети, разброс примерно ±0,1) и считается, как часто каждый
участок остаётся в первой десятке.

In [ ]:
rng = np.random.default_rng(0)
lg = np.log(risk[['H', 'E', 'V']].clip(lower=.05).values)
W = rng.dirichlet([20, 20, 20], 1000)
top10 = np.zeros(len(risk))
for w in W:
    s = (lg * w).sum(1); top10[np.argsort(-s)[:10]] += 1
risk['in_top10'] = top10 / len(W)
st = risk[risk.in_top10 > .05].sort_values('in_top10', ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(st.tj_code + ' · ' + st.site_name.map(lambda n: U.short(n, 36)), 100*st.in_top10,
        color=[U.BLUE if v >= .8 else U.GREY for v in st.in_top10])
for i, v in enumerate(st.in_top10): ax.text(100*v + 1, i, pct(v), va='center', fontsize=8)
ax.set_xlim(0, 112); ax.set_xlabel('% пересчётов, в которых участок в первой десятке')
ax.set_title('Ядро первой десятки не зависит от выбора весов'); plt.show()
U.note(f"{int((risk.in_top10 >= .8).sum())} участков остаются в первой десятке не менее чем в 80 % пересчётов: "
       "приоритеты для вмешательства устойчивы.")

## 7. План следующей кампании отбора проб

Модель и индекс превращаются в конкретный план: какие участки обследовать и зачем.

In [ ]:
plan = pd.read_csv(LAYER / 'sampling_plan/sampling_plan.csv', encoding='utf-8-sig')
summ = json.loads((LAYER / 'sampling_plan/summary.json').read_text(encoding='utf-8'))
g = plan.groupby('group_name', sort=False).agg(Участков=('tj_code', 'size'), Проб=('n_samples', 'sum'),
                                                Детей_в_1_км=('children_1000m', 'sum')).reset_index()
g.columns = ['Группа', 'Участков', 'Проб', 'Детей в 1 км']
U.table(g, nd=0)
fig, ax = plt.subplots(figsize=(7.5, 5))
U.hillshade(ax, (67.3, 36.6, 71.2, 41.1))
ax.scatter(risk.longitude, risk.latitude, s=8, c='#999', zorder=2, label='остальные участки')
pc = [U.RED, U.ORANGE, U.BLUE, U.GREEN]
for (gn, d), c in zip(plan.groupby('group_name', sort=False), pc):
    ax.scatter(d.longitude, d.latitude, s=55, c=c, edgecolor='#222', lw=.6, zorder=3, label=f'{gn} ({len(d)})')
U.map_axes(ax); ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1)); ax.set_title('Участки плана отбора по группам'); plt.show()
U.note(f"{summ['n_sites']} участков, около {summ['n_samples']} проб. В группе первичной оценки модель ожидает "
       f"около {f(summ['group1_expected_strong'], 0)} сильно загрязнённых участков — кампания станет независимой проверкой модели.")

## Выводы

- Из разрозненных PDF и карточек получен проверяемый слой данных: каждое значение ссылается на документ‑источник.
- Модель скрининга по сведениям до анализа находит сильно загрязнённые участки примерно вдвое быстрее случайного
  выбора; результат проверен на географически отложенных участках и не объясняется случайностью.
- Модель надёжно выделяет верхнюю треть участков; среднюю и нижнюю различает слабо. Она задаёт очерёдность, но не
  заменяет лабораторию.
- Индекс риска для детей прозрачен, пересчитывается по формуле и устойчив к выбору весов.
- Следующий шаг — кампания по плану отбора: она проверит модель на новых участках и даст данные для её улучшения.